In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm

import lmm

ModuleNotFoundError: No module named 'lmm'

In [2]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_nrdr_lmm'
!mkdir -p $outfigdir

mkdir: cannot create directory ‘/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_nrdr_lmm’: File exists


In [3]:
adata = sc.read("/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_P21NRDR.h5ad")
adata.X = adata.raw.X
adata

AnnData object with n_obs × n_vars = 31039 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [4]:
# remove mitocondria genes
adata = adata[:,~adata.var.index.str.contains(r'^mt-')]

# remove sex genes
sex_genes = ["Xist", "Uty", "Eif2s3y", "Kdm5d", "Ddx3y"]
adata = adata[:,[g for g in adata.var.index if g not in sex_genes]]

# filter genes
cond = np.ravel((adata.X>0).sum(axis=0)) > 10 # expressed in more than 10 cells
adata = adata[:,cond].copy()
# genes = adata.var.index.values

adata

AnnData object with n_obs × n_vars = 31039 × 16547
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [5]:
np.array(natsorted(np.unique(adata.obs['Age'].values)))

array(['P21', 'P21DR'], dtype='<U5')

In [6]:
cell_abundances = adata.obs.groupby(['Subclass', 'Age']).size().unstack()
cell_abundances
# value_counts()

Age,P21,P21DR
Subclass,,
Astro,884,1258
Endo,74,96
Frem1,42,42
L2/3,2213,4256
L2/3/4,0,214
L4,2173,3758
L5IT,424,787
L5NP,204,291
L5PT,392,762


In [7]:
num_cells_th = 100
uniq_subclasses = cell_abundances[cell_abundances.min(axis=1) > num_cells_th].index.values.astype(str)
uniq_subclasses

array(['Astro', 'L2/3', 'L4', 'L5IT', 'L5NP', 'L5PT', 'L6CT', 'L6IT',
       'Lamp5', 'Micro', 'OD', 'OPC', 'Pvalb', 'Sst', 'Vip'], dtype='<U6')

In [8]:
# adata.obs['cond'] = adata.obs['cond'].apply(lambda x: x.replace('NR', ""))

# sample_labels = adata.obs['Sample'].values
# time_labels = [s[:-1].replace('DR', '') for s in sample_labels]

# adata.obs['sample'] = sample_labels #
# adata.obs['time']   = time_labels

# uniq_samples = natsorted(np.unique(sample_labels))
# nr_samples = [s for s in uniq_samples if "DR" not in s]
# dr_samples = [s for s in uniq_samples if "DR" in s]

# uniq_conds = np.array(natsorted(np.unique(adata.obs['cond'].values)))

# print(uniq_conds)

In [10]:
%%time

tag = 'd251027'
subclass = 'Astro'
time = 'P21'
exp_conds = [time, time+'DR']
# subclass  = 'L2/3'
subclass_cure = subclass.replace('/', '')
offset = 1e-2
scale = 1e4

adatasub = adata[(adata.obs['Age'].isin(exp_conds)) & (adata.obs['Subclass']==subclass)]

# ### test
# adatasub = adatasub[:,:20]
# ### test

genes = adatasub.var.index.values 

obs_fixed = 'Age'
obs_random = 'Sample'
obs = adatasub.obs[[obs_fixed, obs_random]].copy()
obs = obs.dropna()

adatasub = adatasub[obs.index]

output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_{time}_{subclass_cure}_{tag}.csv')

# mat
mat = np.array(adatasub.X.todense())/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

df_res = lmm.run_lmm(mat, genes, obs, obs_fixed, obs_random, output=output, offset=offset)

(2142, 16547) (2142, 2)
(2142, 15992) (2142, 2)
(2142, 10338) (2142, 2)
380 ['Snhg6' 'Kcnb2' 'Bend6' 'Arid5a' '2010300C02Rik' 'Creg2' 'Gpr45' 'Coq10b'
 'Rpl37a' 'Ptprn' 'Tmem198' 'Nyap2' 'Cops9' 'Cntnap5a' 'Map3k19'
 '6030442K20Rik' 'Btg2' 'Gm2000' 'Hsd17b7' 'Cnih3' 'Eprs' 'Rrp15' 'Cd34'
 'Cacna1b' 'Rnf208' 'Ptgds' 'Tmem141' 'Dolpp1' 'Dnm1' 'St6galnac6' 'Rpl12'
 'Rprm' 'Galnt13' 'Nr4a2' 'Ccdc148' 'Gca' 'C1qtnf4' 'Syt13' 'Gm13936'
 'Lpcat4' 'Ckmt1' 'Fbn1' 'Ciao1' 'Polr1b' 'Lamp5' 'Thbd' 'Srxn1' 'Dnmt3b'
 'Dsn1' 'Ppp1r16b' 'Lpin3' 'Cebpb' 'Atp9a' 'Tshz2' 'Atp5e' 'Rps21'
 'Pcsk1n' 'Rpl39' 'Nsdhl' 'Pnck' 'L1cam' 'Emd' 'Col4a6' 'Nxt2' 'Pak3'
 'Gm45022' 'Tmsb4x' 'G530011O06Rik' 'Spry1' 'Pabpc4l' 'Hist2h4' 'Ankrd34a'
 'Tspan2' 'Gm27008' 'Slc6a17' 'Gpr88' 'Plppr4' 'Npnt' 'B230334C09Rik'
 'Rps20' 'Gem' 'Gm11867' 'Lingo2' 'Ccl27a' 'Phf24' 'Tesk1' 'Glipr2'
 'Nr4a3' 'Klf4' 'Susd1' 'Snx30' '8030451A03Rik' 'Adamtsl1' 'Dnajc6' 'Faah'
 'Rps8' 'Dmap1' 'Rimkla' 'Rims3' 'Hpcal4' 'Gm12925' 'Pou3f1' 'Ncdn'

100% 10338/10338 [16:36<00:00, 10.37it/s]


87 ['6030442K20Rik' 'Btg2' 'Rrp15' 'Dnm1' 'Galnt13' 'Nr4a2' 'Syt13' 'Polr1b'
 'Lpin3' 'Cebpb' 'Pcsk1n' 'Nsdhl' 'Pnck' 'Spry1' 'Hist2h4' 'Lingo2'
 'Ccl27a' 'Phf24' 'Klf4' 'Snx30' 'Faah' 'Hpcal4' 'Dlgap3' 'Pink1' 'Zbtb48'
 'Nphp4' 'Cyp51' 'Pus1' '2410131K14Rik' 'Mdfic' 'Tmsb10' 'Aplf' 'Gpr27'
 'Plxnd1' 'Pnmal2' 'Slc17a7' 'Unc45a' 'Pgm2l1' 'P4ha3' 'Plekhb1' 'Ears2'
 'Sprn' 'Shank2' 'Abhd17a' 'Gm16159' 'Junb' 'Gm45812' 'Tmem231' 'Mvd'
 'Gm26759' 'Jph4' 'Tinf2' 'Gjb2' 'Ldlr' 'Gm29521' 'Gm12089' 'Cyfip2'
 'Sowaha' '1700016P03Rik' 'Slc46a1' 'Idi1' 'Hist1h4d' 'Hist1h1e'
 'Hist1h2ac' 'Hmgcr' 'Map1b' 'Zswim6' 'Hmgcs1' 'Fos' 'Kifc2' 'Mapk8ip2'
 'Cacnb3' 'Kcnh3' 'Nr4a1' 'Hes1' 'Gap43' 'Pde10a' 'Dusp1' 'Sik1' 'Crem'
 'Celf4' 'Mbp' 'Sptbn2' 'Gm14966' 'Mamdc2' 'Ppp1r3c' 'Neurl1a']
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_nrdr_lmm/NRDR_DEGs_LMM_P21_Astro_d251027.csv
CPU times: user 16min 33s, sys: 7 s, total: 16min 40s
Wall time: 16min 39s
